# 02 · Build a detector without machine learning
**Level:** beginner · **Time:** 35–45 minutes · **Runs offline on CPU**

**You will:** turn pixels into a foreground mask, find connected outlines, filter noise, produce labeled boxes, and deliberately break a detector's assumptions.

**Prerequisites:** notebook 01; NumPy, Pillow, Matplotlib, and OpenCV from the project requirements. This is a rule-based detector for controlled bright shapes on a dark background. It does not recognize everyday objects.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


## Step 1 — Make a controlled scene
Start with simple conditions so each step can be explained. The three shapes are separated and brighter than the background. The tiny bright dots simulate nuisance detections.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

scene = Image.new("RGB", (420, 220), (25, 30, 35))
painter = ImageDraw.Draw(scene)
painter.rectangle((25, 55, 114, 159), fill=(235, 200, 80))
painter.ellipse((155, 60, 254, 159), fill=(90, 220, 220))
painter.polygon([(340, 40), (285, 165), (395, 165)], fill=(240, 150, 170))
for x, y in [(20, 20), (130, 180), (275, 25), (400, 190)]:
    painter.ellipse((x, y, x + 3, y + 3), fill="white")
plt.figure(figsize=(10, 4))
plt.imshow(scene)
plt.title("Objects plus small bright distractors")
plt.axis("off")
plt.show()


## Step 2 — Separate foreground from background
Convert RGB to grayscale, then mark pixels above a threshold as foreground. Our images come from Pillow and are RGB; OpenCV `imread` would normally return BGR. Choosing the wrong conversion mixes up channel meaning.

**Presenter prompt:** before running, predict whether increasing the threshold will retain more pixels or fewer.


In [ ]:
rgb = np.asarray(scene)
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
THRESHOLD = 100
_, mask = cv2.threshold(gray, THRESHOLD, 255, cv2.THRESH_BINARY)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale intensity")
axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Foreground: intensity > {THRESHOLD}")
for ax in axes:
    ax.axis("off")
plt.show()
print("Foreground pixels:", int((mask > 0).sum()))


## Step 3 — Find outlines and reject tiny regions
`findContours` follows boundaries of foreground regions. `RETR_EXTERNAL` returns only the outer outlines, which is appropriate for these solid shapes. A minimum area removes our small dots. Area filtering can also remove real small objects: a useful rule has a tradeoff.


In [ ]:
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
MIN_AREA = 200
kept_contours = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]
print("Before area filtering:", len(contours), "| after:", len(kept_contours))
print("Retained contour areas:", [round(cv2.contourArea(c), 1) for c in kept_contours])


## Step 4 — Turn geometric rules into predictions
Approximate each outline with fewer vertices. Three corners suggest a triangle; four suggest a rectangle; other rounded contours are called circles **in this restricted dataset**. A four-corner rule also accepts trapezoids, and an ellipse can look like a circle to this rule.

`boundingRect` returns `(x, y, width, height)`; convert to half-open corner coordinates `(x, y, x+width, y+height)`. There is no learned confidence score here.


In [ ]:
def classify_contour(contour):
    perimeter = cv2.arcLength(contour, True)
    polygon = cv2.approxPolyDP(contour, 0.035 * perimeter, True)
    vertices = len(polygon)
    return "triangle" if vertices == 3 else "rectangle" if vertices == 4 else "circle-like"

def detect_bright_shapes(pil_image, threshold=100, min_area=200):
    gray_image = cv2.cvtColor(np.asarray(pil_image), cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray_image, threshold, 255, cv2.THRESH_BINARY)
    outlines, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes, labels = [], []
    for contour in outlines:
        if cv2.contourArea(contour) < min_area:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        boxes.append([x, y, x + w, y + h])
        labels.append(classify_contour(contour))
    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4), labels, binary

predicted_boxes, predicted_names, _ = detect_bright_shapes(scene)
show_boxes(scene, predicted_boxes, predicted_names, title="A complete rule-based detection pipeline")
plt.show()
print(list(zip(predicted_names, predicted_boxes.astype(int).tolist())))


## Step 5 — Break the assumptions, one change at a time
Dim objects fall below the intensity threshold. Touching bright objects become one connected region. These failures illustrate why background contrast and separation are assumptions, not guaranteed properties of images.


In [ ]:
dim_scene = Image.fromarray((np.asarray(scene).astype(float) * 0.40).astype(np.uint8))
touching_scene = Image.new("RGB", scene.size, (25, 30, 35))
touch_draw = ImageDraw.Draw(touching_scene)
touch_draw.rectangle((70, 50, 180, 165), fill=(220, 220, 100))
touch_draw.ellipse((155, 60, 265, 170), fill=(100, 220, 220))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, picture, title in zip(axes, [scene, dim_scene, touching_scene],
                              ["Original", "Dim lighting", "Two touching objects"]):
    found, labels, _ = detect_bright_shapes(picture)
    show_boxes(picture, found, labels, ax=ax, title=f"{title}: {len(found)} detections")
plt.tight_layout()
plt.show()


**Observe:** changing a threshold can recover dim foreground, but it cannot split two objects that share one connected mask. A successful demonstration should include a failure case and explain its cause.

## Try it yourself
1. Sweep thresholds 30, 60, 100, 180, 240; record the number of detected shapes in the original scene.
2. Repair the dim scene by changing one parameter. Does the same setting work on a bright background?
3. Set `min_area=0`. Which extra boxes appear, and why?


## Solutions — reveal after trying
Counts depend on grayscale intensities and region areas. A count of three alone does not prove that the three boxes are correct: always inspect them.


In [ ]:
for value in [30, 60, 100, 180, 240]:
    found, labels, _ = detect_bright_shapes(scene, threshold=value)
    print(f"threshold={value:3d}: {len(found)} detections", labels)
repaired, labels, _ = detect_bright_shapes(dim_scene, threshold=50)
show_boxes(dim_scene, repaired, labels, title="Lower threshold recovers dim shapes")
plt.show()
noisy_boxes, noisy_labels, _ = detect_bright_shapes(scene, min_area=0)
show_boxes(scene, noisy_boxes, noisy_labels, title="Removing the area filter keeps small dots")
plt.show()


## Recap and next step
You built an image → mask → contours → rules → labeled boxes pipeline. Its assumptions are easy to explain and easy to violate. Learned detectors replace many manual rules with features learned from varied labeled examples.

**Next:** [03 · IoU and non-maximum suppression](03_IoU_and_Non_Maximum_Suppression.ipynb).

**References:** [OpenCV contours tutorial](https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html), [contour features](https://docs.opencv.org/4.x/dd/d49/tutorial_py_contour_features.html).
